<a href="https://colab.research.google.com/github/muajnstu/Customer_Churn_Prediction/blob/main/Employee_Attrition_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, NuSVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, roc_auc_score

In [ ]:
def train_and_evaluate_models(models, X_train, X_test, y_train, y_test):
    results = {}
    for name, model in models.items():
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred)

        results[name] = {
            'accuracy': accuracy,
            'precision': precision,
            'f1_score': f1,
            'recall': recall,
            'auc': auc
        }

        print(f"{name} metrics:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  AUC: {auc:.4f}")

    return results

# Data Loading

In [ ]:
liza=pd.read_csv('https://raw.githubusercontent.com/muajnstu/ML-Datasets/refs/heads/main/employee_attrition_data.csv')

In [ ]:
liza

,Employee_ID,Age,Gender,Department,Job_Title,Years_at_Company,Satisfaction_Level,Average_Monthly_Hours,Promotion_Last_5Years,Salary,Attrition
0,0,27,Male,Marketing,Manager,9,0.586251,151,0,60132,0
1,1,53,Female,Sales,Engineer,10,0.261161,221,1,79947,0
2,2,59,Female,Marketing,Analyst,8,0.304382,184,0,46958,1
3,3,42,Female,Engineering,Manager,1,0.480779,242,0,40662,0
4,4,44,Female,Sales,Engineer,10,0.636244,229,1,74307,0
...,...,...,...,...,...,...,...,...,...,...,...
995,995,39,Female,HR,HR Specialist,3,0.377435,239,0,71403,0
996,996,50,Male,Engineering,Manager,1,0.431152,154,0,30181,1
997,997,52,Male,Engineering,Analyst,3,0.647102,206,0,64143,0
998,998,37,Female,HR,HR Specialist,2,0.304813,241,0,74383,1


In [ ]:
liza.duplicated().sum()
print(liza.duplicated().sum())

0


In [ ]:
liza.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Employee_ID            1000 non-null   int64  
 1   Age                    1000 non-null   int64  
 2   Gender                 1000 non-null   object 
 3   Department             1000 non-null   object 
 4   Job_Title              1000 non-null   object 
 5   Years_at_Company       1000 non-null   int64  
 6   Satisfaction_Level     1000 non-null   float64
 7   Average_Monthly_Hours  1000 non-null   int64  
 8   Promotion_Last_5Years  1000 non-null   int64  
 9   Salary                 1000 non-null   int64  
 10  Attrition              1000 non-null   int64  
dtypes: float64(1), int64(7), object(3)
memory usage: 86.1+ KB


In [ ]:
liza['Attrition'].value_counts()

,count
Attrition,
0,505
1,495


In [ ]:
for col in liza.columns:
    print(f"Unique values for column '{col}':")
    print(liza[col].unique())
    print("\n")

Unique values for column 'Employee_ID':
[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 23

# Data Cleaning

1. Encoding
2. Drop
3. Correlation

In [ ]:
gender_mapping = {'Male': 0, 'Female': 1}
department_mapping = {'Marketing': 0, 'Sales': 1, 'Engineering': 2, 'Finance': 3, 'HR': 4}
job_title_mapping = {'Manager': 0, 'Engineer': 1, 'Analyst': 2, 'HR Specialist': 3, 'Accountant': 4}
liza['Gender_encoded'] = liza['Gender'].map(gender_mapping)
liza['Department_encoded'] = liza['Department'].map(department_mapping)
liza['Job_Title_encoded'] = liza['Job_Title'].map(job_title_mapping)
print(liza[['Gender', 'Gender_encoded', 'Department', 'Department_encoded', 'Job_Title', 'Job_Title_encoded']].head())
for col in ['Gender_encoded', 'Department_encoded', 'Job_Title_encoded']:
    print(f"Unique values for column '{col}':")
    print(liza[col].unique())
    print("\n")


   Gender  Gender_encoded   Department  Department_encoded Job_Title  \
0    Male               0    Marketing                   0   Manager   
1  Female               1        Sales                   1  Engineer   
2  Female               1    Marketing                   0   Analyst   
3  Female               1  Engineering                   2   Manager   
4  Female               1        Sales                   1  Engineer   

   Job_Title_encoded  
0                  0  
1                  1  
2                  2  
3                  0  
4                  1  
Unique values for column 'Gender_encoded':
[0 1]


Unique values for column 'Department_encoded':
[0 1 2 3 4]


Unique values for column 'Job_Title_encoded':
[0 1 2 3 4]




In [ ]:
liza.drop(['Employee_ID','Gender', 'Department', 'Job_Title'], axis=1, inplace=True)

In [ ]:
liza.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    1000 non-null   int64  
 1   Years_at_Company       1000 non-null   int64  
 2   Satisfaction_Level     1000 non-null   float64
 3   Average_Monthly_Hours  1000 non-null   int64  
 4   Promotion_Last_5Years  1000 non-null   int64  
 5   Salary                 1000 non-null   int64  
 6   Attrition              1000 non-null   int64  
 7   Gender_encoded         1000 non-null   int64  
 8   Department_encoded     1000 non-null   int64  
 9   Job_Title_encoded      1000 non-null   int64  
dtypes: float64(1), int64(9)
memory usage: 78.3 KB


In [ ]:
liza_corr = liza.corr(method='spearman')
liza_corr

,Age,Years_at_Company,Satisfaction_Level,Average_Monthly_Hours,Promotion_Last_5Years,Salary,Attrition,Gender_encoded,Department_encoded,Job_Title_encoded
Age,1.000000,0.017408,0.022800,-0.028880,0.038948,-0.016899,-0.008769,-0.037304,-0.035719,0.030140
Years_at_Company,0.017408,1.000000,-0.033851,-0.060330,-0.010303,0.036339,0.003381,-0.021234,0.056427,0.023812
Satisfaction_Level,0.022800,-0.033851,1.000000,-0.008978,0.003528,-0.028093,-0.009683,-0.053268,-0.011513,0.024488
Average_Monthly_Hours,-0.028880,-0.060330,-0.008978,1.000000,-0.032078,-0.053435,-0.024612,0.064077,0.002235,-0.004709
Promotion_Last_5Years,0.038948,-0.010303,0.003528,-0.032078,1.000000,0.017535,0.017728,0.007668,-0.042797,0.006831
Salary,-0.016899,0.036339,-0.028093,-0.053435,0.017535,1.000000,-0.037989,0.071726,-0.040775,-0.025963
Attrition,-0.008769,0.003381,-0.009683,-0.024612,0.017728,-0.037989,1.000000,0.033884,0.052450,0.013803
Gender_encoded,-0.037304,-0.021234,-0.053268,0.064077,0.007668,0.071726,0.033884,1.000000,0.003274,-0.033329
Department_encoded,-0.035719,0.056427,-0.011513,0.002235,-0.042797,-0.040775,0.052450,0.003274,1.000000,0.007794
Job_Title_encoded,0.030140,0.023812,0.024488,-0.004709,0.006831,-0.025963,0.013803,-0.033329,0.007794,1.000000


# Model Building

In [ ]:
X = liza.drop('Attrition', axis=1)
y = liza['Attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (800, 9)
Shape of X_test: (200, 9)
Shape of y_train: (800,)
Shape of y_test: (200,)


In [ ]:
models = {
    "LogisticRegression": LogisticRegression(),
    "SVC": SVC(),
    "NuSVC": NuSVC(),
    "LinearSVC": LinearSVC(),
    "KNeighborsClassifier": KNeighborsClassifier(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "ExtraTreeClassifier": ExtraTreeClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "GradientBoostingClassifier": GradientBoostingClassifier(),
    "AdaBoostClassifier": AdaBoostClassifier(),
    "ExtraTreesClassifier": ExtraTreesClassifier(),
    "GaussianNB": GaussianNB(),
    "BernoulliNB": BernoulliNB(),
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "QuadraticDiscriminantAnalysis": QuadraticDiscriminantAnalysis()
}

# Evaluation

In [ ]:
model_results = train_and_evaluate_models(models, X_train, X_test, y_train, y_test)


Training LogisticRegression...
LogisticRegression metrics:
  Accuracy: 0.4500
  Precision: 0.4318
  F1 Score: 0.4086
  Recall: 0.3878
  AUC: 0.4488
Training SVC...
SVC metrics:
  Accuracy: 0.4800
  Precision: 0.4625
  F1 Score: 0.4157
  Recall: 0.3776
  AUC: 0.4780
Training NuSVC...
NuSVC metrics:
  Accuracy: 0.4150
  Precision: 0.4275
  F1 Score: 0.4891
  Recall: 0.5714
  AUC: 0.4181
Training LinearSVC...
LinearSVC metrics:
  Accuracy: 0.4550
  Precision: 0.4382
  F1 Score: 0.4171
  Recall: 0.3980
  AUC: 0.4539
Training KNeighborsClassifier...


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


KNeighborsClassifier metrics:
  Accuracy: 0.5100
  Precision: 0.5000
  F1 Score: 0.4896
  Recall: 0.4796
  AUC: 0.5094
Training DecisionTreeClassifier...
DecisionTreeClassifier metrics:
  Accuracy: 0.4600
  Precision: 0.4510
  F1 Score: 0.4600
  Recall: 0.4694
  AUC: 0.4602
Training ExtraTreeClassifier...
ExtraTreeClassifier metrics:
  Accuracy: 0.4850
  Precision: 0.4737
  F1 Score: 0.4663
  Recall: 0.4592
  AUC: 0.4845
Training RandomForestClassifier...
RandomForestClassifier metrics:
  Accuracy: 0.4700
  Precision: 0.4500
  F1 Score: 0.4045
  Recall: 0.3673
  AUC: 0.4680
Training GradientBoostingClassifier...
GradientBoostingClassifier metrics:
  Accuracy: 0.4350
  Precision: 0.4227
  F1 Score: 0.4205
  Recall: 0.4184
  AUC: 0.4347
Training AdaBoostClassifier...
AdaBoostClassifier metrics:
  Accuracy: 0.4750
  Precision: 0.4507
  F1 Score: 0.3787
  Recall: 0.3265
  AUC: 0.4721
Training ExtraTreesClassifier...
ExtraTreesClassifier metrics:
  Accuracy: 0.4400
  Precision: 0.4205
  F1 